# Mask RCNN Training - Road Damage Detection (RDD)

Complete pipeline for training Mask RCNN model to detect road cracks and potholes.

**Dataset**: RDD (5,770 images, 2 classes)
**Framework**: Detectron2 (PyTorch)
**Model**: Mask RCNN with ResNet50-FPN

---

## 1. Setup & Environment

In [1]:
# Install required packages
import subprocess
import sys

# Install detectron2
print("Installing detectron2...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "detectron2", "-q"])
print("✓ Detectron2 installed")

Installing detectron2...


CalledProcessError: Command '['c:\\Users\\Admin\\anaconda3\\python.exe', '-m', 'pip', 'install', 'detectron2', '-q']' returned non-zero exit status 1.

In [ ]:
# Import libraries
import os
import cv2
import numpy as np
import json
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from tqdm import tqdm

# Detectron2 imports
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, default_argument_parser, launch
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.structures import BoxMode
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.engine import DefaultPredictor
from detectron2.visualization import Visualizer

print("✓ All libraries imported successfully")

# Check CUDA
print(f"\nCUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
# Configuration
DATA_DIR = "Road_Patches_Cracks_Segmentation_Datasets"
OUTPUT_DIR = "output_mask_rcnn"
ANALYSIS_DIR = "dataset_analysis"

NUM_CLASSES = 2
CLASS_NAMES = ["Crack", "Pothole"]
IMG_SIZE = 432

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ANALYSIS_DIR, exist_ok=True)

print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Classes: {CLASS_NAMES}")

# Plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Data Loading & Conversion

In [ ]:
def parse_yolo_labels(label_file, img_width, img_height):
    """Convert YOLO polygon format to Detectron2 format"""
    annotations = []

    try:
        with open(label_file, 'r') as f:
            lines = f.readlines()
    except:
        return annotations

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:  # Need at least class_id + 2 points (4 coords)
            continue

        class_id = int(parts[0])
        coords = list(map(float, parts[1:]))

        # Convert normalized coords to pixel coords
        polygon = []
        for i in range(0, len(coords), 2):
            if i + 1 < len(coords):
                x = coords[i] * img_width
                y = coords[i + 1] * img_height
                polygon.extend([x, y])

        if len(polygon) >= 6:  # At least 3 points
            annotations.append({
                "category_id": class_id,
                "segmentation": [polygon],
                "bbox_mode": BoxMode.XYWH_ABS
            })

    return annotations

print("✓ YOLO parser function defined")

In [ ]:
def get_dataset_dicts(split="train"):
    """Load dataset in Detectron2 format"""
    dataset_dicts = []

    img_dir = os.path.join(DATA_DIR, split, "images")
    label_dir = os.path.join(DATA_DIR, split, "labels")

    if not os.path.exists(img_dir):
        print(f"Warning: {img_dir} not found")
        return dataset_dicts

    img_files = sorted(os.listdir(img_dir))

    for img_file in tqdm(img_files, desc=f"Loading {split}", leave=False):
        img_path = os.path.join(img_dir, img_file)
        label_file = os.path.join(label_dir, img_file.rsplit('.', 1)[0] + '.txt')

        # Read image to get dimensions
        img = cv2.imread(img_path)
        if img is None:
            continue

        height, width = img.shape[:2]

        # Parse labels
        annotations = parse_yolo_labels(label_file, width, height)

        # Calculate bounding boxes from polygons
        objs = []
        for ann in annotations:
            polygon = ann["segmentation"][0]

            # Get bbox from polygon points
            xs = [polygon[i] for i in range(0, len(polygon), 2)]
            ys = [polygon[i] for i in range(1, len(polygon), 2)]

            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)

            objs.append({
                "bbox": [x_min, y_min, x_max - x_min, y_max - y_min],
                "bbox_mode": BoxMode.XYWH_ABS,
                "category_id": ann["category_id"],
                "segmentation": ann["segmentation"]
            })

        record = {
            "file_name": img_path,
            "image_id": len(dataset_dicts),
            "height": height,
            "width": width,
            "annotations": objs
        }

        dataset_dicts.append(record)

    print(f"Loaded {len(dataset_dicts)} images from {split}")
    return dataset_dicts

print("✓ Dataset loader function defined")

In [ ]:
def register_datasets():
    """Register train, val, test datasets"""
    for split in ["train", "valid", "test"]:
        if split in DatasetCatalog.list():
            DatasetCatalog.remove(f"rdd_{split}")
        
        DatasetCatalog.register(f"rdd_{split}", lambda s=split: get_dataset_dicts(s))
        MetadataCatalog.get(f"rdd_{split}").set(thing_classes=CLASS_NAMES)

    print("✓ Datasets registered")

# Register datasets
register_datasets()

In [ ]:
# Load dataset info
train_dicts = get_dataset_dicts("train")
valid_dicts = get_dataset_dicts("valid")
test_dicts = get_dataset_dicts("test")

print(f"\nDataset Summary:")
print(f"  Train: {len(train_dicts)} images")
print(f"  Valid: {len(valid_dicts)} images")
print(f"  Test:  {len(test_dicts)} images")
print(f"  Total: {len(train_dicts) + len(valid_dicts) + len(test_dicts)} images")

## 3. Dataset Analysis & Visualization

In [ ]:
def analyze_dataset_dicts(dataset_dicts):
    """Analyze dataset statistics"""
    stats = {
        "total_images": len(dataset_dicts),
        "images_with_objects": 0,
        "total_objects": 0,
        "objects_per_class": defaultdict(int),
        "object_areas": []
    }

    for record in dataset_dicts:
        if len(record["annotations"]) > 0:
            stats["images_with_objects"] += 1
            stats["total_objects"] += len(record["annotations"])

            for ann in record["annotations"]:
                cls_id = ann["category_id"]
                cls_name = CLASS_NAMES[cls_id]
                stats["objects_per_class"][cls_name] += 1

                # Calculate area
                bbox = ann["bbox"]
                area = bbox[2] * bbox[3]  # width * height
                stats["object_areas"].append(area)

    return stats

# Analyze
train_stats = analyze_dataset_dicts(train_dicts)
valid_stats = analyze_dataset_dicts(valid_dicts)
test_stats = analyze_dataset_dicts(test_dicts)

print("\n=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

for split, stats in [("Train", train_stats), ("Valid", valid_stats), ("Test", test_stats)]:
    print(f"\n{split}:")
    print(f"  Total Images: {stats['total_images']}")
    print(f"  Images with Objects: {stats['images_with_objects']} ({100*stats['images_with_objects']/stats['total_images']:.1f}%)")
    print(f"  Total Objects: {stats['total_objects']}")
    print(f"  Objects per Class:")
    for cls, count in sorted(stats['objects_per_class'].items()):
        print(f"    - {cls}: {count}")
    if stats['object_areas']:
        print(f"  Object Area: {np.mean(stats['object_areas']):.0f} ± {np.std(stats['object_areas']):.0f} px²")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Across all splits
all_classes = defaultdict(int)
for stats in [train_stats, valid_stats, test_stats]:
    for cls, count in stats["objects_per_class"].items():
        all_classes[cls] += count

classes = list(all_classes.keys())
counts = list(all_classes.values())

# Bar plot
axes[0].bar(classes, counts, color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Class Distribution (All Images)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(counts):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart
colors = ['#FF6B6B', '#4ECDC4']
axes[1].pie(counts, labels=classes, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Class Proportion', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(ANALYSIS_DIR, "class_distribution.png"), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: class_distribution.png")

In [ ]:
# Visualize dataset split
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

splits = ["Train", "Valid", "Test"]
total_images = [train_stats["total_images"], valid_stats["total_images"], test_stats["total_images"]]
with_objects = [train_stats["images_with_objects"], valid_stats["images_with_objects"], test_stats["images_with_objects"]]

# Images per split
x = np.arange(len(splits))
width = 0.35

axes[0].bar(x - width/2, total_images, width, label='Total Images', color='#95E1D3')
axes[0].bar(x + width/2, with_objects, width, label='With Objects', color='#F38181')
axes[0].set_title('Dataset Split Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Images')
axes[0].set_xticks(x)
axes[0].set_xticklabels(splits)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Percentage with objects
percentages = [100 * with_objects[i] / total_images[i] for i in range(len(splits))]
axes[1].bar(splits, percentages, color=['#95E1D3', '#F38181', '#AAF683'])
axes[1].set_title('% Images with Objects', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_ylim([0, 100])
axes[1].grid(axis='y', alpha=0.3)

for i, v in enumerate(percentages):
    axes[1].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(ANALYSIS_DIR, "dataset_split.png"), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: dataset_split.png")

## 4. Model Configuration & Training

In [ ]:
def setup_config(output_dir):
    """Configure Mask RCNN"""
    cfg = get_cfg()
    cfg.merge_from_file("detectron2://configs/COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

    # Dataset
    cfg.DATASETS.TRAIN = ("rdd_train",)
    cfg.DATASETS.TEST = ("rdd_valid",)

    # Dataloader
    cfg.DATALOADER.NUM_WORKERS = 4

    # Model
    cfg.MODEL.BACKBONE.FREEZE_AT = 2
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512

    # Solver
    cfg.SOLVER.IMS_PER_BATCH = 4
    cfg.SOLVER.BASE_LR = 0.001
    cfg.SOLVER.MAX_ITER = 2000
    cfg.SOLVER.STEPS = (1000, 1500)
    cfg.SOLVER.GAMMA = 0.1
    cfg.SOLVER.WARMUP_ITERS = 100

    # Checkpoint
    cfg.SOLVER.CHECKPOINT_PERIOD = 500

    # Test
    cfg.TEST.EVAL_PERIOD = 500
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
    cfg.MODEL.MASK_ON = True

    # Output
    cfg.OUTPUT_DIR = output_dir

    return cfg

print("✓ Config function defined")

In [ ]:
class CocoTrainer(DefaultTrainer):
    """Custom trainer with COCO evaluation"""

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, output_dir=output_folder)

print("✓ CocoTrainer class defined")

In [ ]:
print("="*60)
print("MASK RCNN TRAINING - ROAD DAMAGE DETECTION")
print("="*60)
print(f"\nDataset: {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Classes: {CLASS_NAMES}")

# Setup config
cfg = setup_config(OUTPUT_DIR)

print("\n" + "="*60)
print("MODEL CONFIGURATION")
print("="*60)
print(f"Backbone: ResNet50-FPN")
print(f"Training Images: {len(train_dicts)}")
print(f"Validation Images: {len(valid_dicts)}")
print(f"Batch Size: {cfg.SOLVER.IMS_PER_BATCH}")
print(f"Max Iterations: {cfg.SOLVER.MAX_ITER}")
print(f"Base LR: {cfg.SOLVER.BASE_LR}")
print(f"LR Schedule: Step decay at iter {cfg.SOLVER.STEPS}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Train model
print("\n" + "="*60)
print("STARTING TRAINING...")
print("="*60)

trainer = CocoTrainer(cfg)
trainer.resume_or_load(resume=False)

start_time = datetime.now()
trainer.train()
end_time = datetime.now()

duration = (end_time - start_time).total_seconds() / 3600
print(f"\n✓ Training completed in {duration:.2f} hours")
print(f"✓ Model saved to: {OUTPUT_DIR}")

## 5. Model Evaluation

In [ ]:
def evaluate_dataset(dataset_name):
    """Evaluate on a dataset"""
    cfg = get_cfg()
    cfg.merge_from_file("detectron2://configs/COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

    cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.WEIGHTS = os.path.join(OUTPUT_DIR, "model_final.pth")
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
    cfg.MODEL.MASK_ON = True

    cfg.DATASETS.TEST = (dataset_name,)
    cfg.DATALOADER.NUM_WORKERS = 4
    cfg.OUTPUT_DIR = OUTPUT_DIR

    trainer = DefaultTrainer(cfg)
    trainer.resume_or_load(resume=False)

    evaluator = COCOEvaluator(dataset_name, output_dir=OUTPUT_DIR)
    val_loader = build_detection_test_loader(cfg, dataset_name)
    results = inference_on_dataset(trainer.model, val_loader, evaluator)

    return results

print("✓ Evaluation function defined")

In [ ]:
# Check if model exists
model_path = os.path.join(OUTPUT_DIR, "model_final.pth")

if os.path.exists(model_path):
    print("="*60)
    print("MODEL EVALUATION")
    print("="*60)

    print("\nEvaluating on Validation Set...")
    val_results = evaluate_dataset("rdd_valid")

    print(f"\n{'='*60}")
    print("VALIDATION RESULTS")
    print(f"{'='*60}")

    if "bbox" in val_results:
        print("\nBounding Box Metrics:")
        print(f"  AP: {val_results['bbox']['AP']:.4f}")
        print(f"  AP50: {val_results['bbox']['AP50']:.4f}")
        print(f"  AP75: {val_results['bbox']['AP75']:.4f}")
        print(f"  AR: {val_results['bbox']['AR']:.4f}")

    if "segm" in val_results:
        print("\nSegmentation Metrics:")
        print(f"  AP: {val_results['segm']['AP']:.4f}")
        print(f"  AP50: {val_results['segm']['AP50']:.4f}")
        print(f"  AP75: {val_results['segm']['AP75']:.4f}")
        print(f"  AR: {val_results['segm']['AR']:.4f}")
else:
    print(f"⚠️  Model not found at {model_path}")
    print("Please train the model first by running the training cell above.")

## 6. Inference & Visualization

In [ ]:
def setup_predictor(model_path):
    """Setup predictor with trained model"""
    cfg = get_cfg()
    cfg.merge_from_file("detectron2://configs/COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

    cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.WEIGHTS = model_path
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
    cfg.MODEL.MASK_ON = True

    predictor = DefaultPredictor(cfg)
    return predictor, cfg

def predict_image(predictor, image_path):
    """Predict on single image"""
    img = cv2.imread(image_path)
    if img is None:
        return None

    outputs = predictor(img)
    return img, outputs

def visualize_predictions(img, outputs, save_path=None):
    """Visualize predictions"""
    v = Visualizer(img[:, :, ::-1], MetadataCatalog.get("rdd_train"), scale=1.2)
    v = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    vis_img = v.get_image()[:, :, ::-1]

    if save_path:
        cv2.imwrite(save_path, vis_img)

    return vis_img

print("✓ Inference functions defined")

In [ ]:
# Setup predictor
if os.path.exists(model_path):
    print(f"Loading model from: {model_path}")
    predictor, cfg = setup_predictor(model_path)
    print("✓ Model loaded successfully")
else:
    print(f"⚠️  Model not found at {model_path}")

In [ ]:
# Single image prediction
if os.path.exists(model_path):
    val_img_dir = os.path.join(DATA_DIR, "valid", "images")
    sample_images = [f for f in os.listdir(val_img_dir) if f.lower().endswith(('.jpg', '.png'))]

    if sample_images:
        sample_path = os.path.join(val_img_dir, sample_images[0])

        print(f"\nPredicting on: {sample_images[0]}")
        pred_result = predict_image(predictor, sample_path)

        if pred_result:
            img, outputs = pred_result
            instances = outputs["instances"]

            print(f"Detections: {len(instances)}")

            if len(instances) > 0:
                pred_classes = instances.pred_classes.cpu().numpy()
                scores = instances.scores.cpu().numpy()

                for i, (cls, score) in enumerate(zip(pred_classes, scores), 1):
                    print(f"  {i}. {CLASS_NAMES[cls]}: {score:.3f}")

            # Visualize
            vis_img = visualize_predictions(img, outputs)
            plt.figure(figsize=(12, 8))
            plt.imshow(vis_img)
            plt.title(f"Predictions: {sample_images[0]}", fontsize=14, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, "sample_prediction.jpg"), dpi=150, bbox_inches='tight')
            plt.show()

            print(f"\n✓ Saved: {os.path.join(OUTPUT_DIR, 'sample_prediction.jpg')}")

In [ ]:
# Batch prediction on validation set
if os.path.exists(model_path):
    val_img_dir = os.path.join(DATA_DIR, "valid", "images")
    val_output_dir = os.path.join(OUTPUT_DIR, "predictions_valid")
    os.makedirs(val_output_dir, exist_ok=True)

    image_files = [f for f in os.listdir(val_img_dir) if f.lower().endswith(('.jpg', '.png'))]

    print(f"\nProcessing {len(image_files)} validation images...")

    results_summary = {
        "total_images": len(image_files),
        "images_with_detection": 0,
        "total_detections": 0,
        "detections_by_class": {cls: 0 for cls in CLASS_NAMES}
    }

    for idx, img_file in enumerate(tqdm(image_files[:20], desc="Predicting")):
        img_path = os.path.join(val_img_dir, img_file)
        pred_result = predict_image(predictor, img_path)

        if pred_result is None:
            continue

        img, outputs = pred_result
        num_detections = len(outputs["instances"])

        if num_detections > 0:
            results_summary["images_with_detection"] += 1
            results_summary["total_detections"] += num_detections

            pred_classes = outputs["instances"].pred_classes.cpu().numpy()
            for cls in pred_classes:
                results_summary["detections_by_class"][CLASS_NAMES[cls]] += 1

            # Visualize
            vis_img = visualize_predictions(img, outputs)
            output_file = os.path.join(val_output_dir, f"pred_{img_file}")
            cv2.imwrite(output_file, vis_img)

    print(f"\n{'='*60}")
    print("PREDICTION SUMMARY (First 20 Images)")
    print(f"{'='*60}")
    print(f"Images Processed: 20")
    print(f"Images with Detection: {results_summary['images_with_detection']}")
    print(f"Total Detections: {results_summary['total_detections']}")
    for cls, count in results_summary['detections_by_class'].items():
        print(f"  - {cls}: {count}")
    print(f"\nVisualized results saved to: {val_output_dir}")

## Summary

In [ ]:
print("\n" + "="*60)
print("MASK RCNN TRAINING COMPLETE")
print("="*60)

print(f"\n📊 Output Files:")
if os.path.exists(OUTPUT_DIR):
    for file in os.listdir(OUTPUT_DIR):
        file_path = os.path.join(OUTPUT_DIR, file)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path) / (1024**2)  # Convert to MB
            print(f"  ✓ {file} ({size:.1f} MB)" if size > 1 else f"  ✓ {file}")

print(f"\n📈 Analysis Files:")
if os.path.exists(ANALYSIS_DIR):
    for file in os.listdir(ANALYSIS_DIR):
        print(f"  ✓ {file}")

print(f"\n🎯 Next Steps:")
print(f"  1. Review evaluation metrics")
print(f"  2. Check prediction visualizations in {val_output_dir}")
print(f"  3. Fine-tune hyperparameters if needed")
print(f"  4. Deploy model for production use")

print(f"\n✓ All done! 🚀")